# **肌電訊號量測實作單元(一)：大肌肉的你有多會放電？一量就知道啦！**



> 這次的實驗會帶大家使用最簡單的弦波，透過疊加來改變時域的波形


> 再透過濾波器來試著還原波形，下面就讓鐵克一步一步，帶大家體驗濾波器的神奇之處吧！






# 0. 先收錄一些會用到的公式和函式吧！

In [1]:
#@title
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from scipy.fft import fft, fftfreq
from scipy import signal
import plotly.graph_objects as go
from scipy.signal import butter, lfilter

def butter_bandpass(low_cut, high_cut, fs, order=5):
    """
    Design band pass filter.

    Args:
        - low_cut  (float) : the low cutoff frequency of the filter.
        - high_cut (float) : the high cutoff frequency of the filter.
        - fs       (float) : the sampling rate.
        - order      (int) : order of the filter, by default defined to 5.
    """
    # calculate the Nyquist frequency
    nyq = 0.5 * fs

    # design filter
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype='band')

    # returns the filter coefficients: numerator and denominator
    return b, a 

def BandPassFilter(data, lowcut, highcut):
    fs = 1000
    order =5
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

def super_fft(data):
  # Number of sample points
  N = int(data.size)
  # sample spacing
  T = 0.001
  x = np.linspace(0.0, N*T, N, endpoint=False)
  yf = fft(data)
  xf = fftfreq(N, T)[1:N//2] 
  yf_abs1 = np.abs(yf[1:N//2]) 
  fig = go.Figure()
  fig.add_trace(go.Scatter(x=xf, y=2.0/N * yf_abs1))
  fig.update_layout(
    title="Frequency Information of the Data",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
  fig.show()

def LowPassFilter(data, freq):
  sos = signal.butter(10, freq, 'lp', fs=1000, output='sos')
  filtered = signal.sosfilt(sos, data)
  return filtered
def HighPassFilter(data, freq):
  sos = signal.butter(10, freq, 'hp', fs=1000, output='sos')
  filtered = signal.sosfilt(sos, data)
  return filtered

# 1. 先產生3個弦波，頻率分別落在10Hz, 60Hz, 120Hz。

In [ ]:
t = np.linspace(0, 1, 1000, False)  # 1 second
signa1_10Hz = np.sin(2*np.pi*10*t)
signa1_60Hz = np.sin(2*np.pi*60*t)
signa1_120Hz = np.sin(2*np.pi*120*t)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True)

ax1.plot(t, signa1_10Hz)
ax1.set_title('10 Hz sinusoids')
ax1.axis([0, 1, -2, 2])

ax2.plot(t, signa1_60Hz)
ax2.set_title('60 Hz sinusoids')
ax2.axis([0, 1, -2, 2])

ax3.plot(t, signa1_120Hz)
ax3.set_title('120 Hz sinusoids')
ax3.axis([0, 1, -2, 2])

ax3.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

In [ ]:
#試著自己將signa1_10Hz、signa1_60Hz、signa1_120Hz一個一個放進super_fft看看吧！
super_fft(signa1_120Hz) 

# 2. 將弦波各自疊加，自製更複雜的訊號來進行後續濾波吧！

In [6]:
signa1_1060 = signa1_10Hz + signa1_60Hz
signa1_60120 = signa1_60Hz + signa1_120Hz 
signal_mix = signa1_10Hz + signa1_60Hz + signa1_120Hz 

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True)

ax1.plot(t, signa1_1060)
ax1.set_title('10 Hz sinusoids + 60 Hz sinusoids')
ax1.axis([0, 1, -2, 2])

ax2.plot(t, signa1_60120)
ax2.set_title('60 Hz sinusoids + 120 Hz sinusoids')
ax2.axis([0, 1, -2, 2])

ax3.plot(t, signal_mix)
ax3.set_title('10 Hz sinusoids + 60 Hz sinusoids + 120 Hz sinusoids')
ax3.axis([0, 1, -2, 2])

ax3.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

In [ ]:
#試著自己將signa1_1060、signal_60120、signal_mix一個一個放進super_fft看看吧！
super_fft(signal_mix) 

# 3. 拿signal_mix來練習高通濾波器吧！



> 高通濾波器的濾波頻率，會決定該頻率以下的波形被濾除唷！



In [ ]:
#試著把濾波頻率往下調成100、70、20、1看看吧 那如果調成150又會如何呢？
filtered1 = HighPassFilter(signal_mix, 1) 
fig, (ax1) = plt.subplots(1, 1, sharex=True)

ax1.plot(t, filtered1)
ax1.set_title('Data after HighPassFilter')
ax1.axis([0, 1, -2, 2])
ax1.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

super_fft(filtered1) 

# 4. 拿signal_mix來練習低通濾波器吧！

> 帶通濾波器的濾波頻率，會決定該頻率以上的波形被濾除唷！

In [ ]:
#試著把濾波頻率往上調成20、50、100、150看看吧 那如果調成5又會如何呢？
filtered2 = LowPassFilter(signal_mix, 150) 
fig, (ax1) = plt.subplots(1, 1, sharex=True)

ax1.plot(t, filtered2)
ax1.set_title('Data after LowPassFilter')
ax1.axis([0, 1, -2, 2])
ax1.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

super_fft(filtered2) 

# 5. 拿signal_1060120來練習帶通濾波器吧！

> 帶通濾波器的濾波頻率，會決定該頻率以外的波形被濾除唷！

In [ ]:
#試著把濾波頻率調成(5,15)、(40、80)、 (100、140)看看吧
filtered3 = BandPassFilter(signal_mix, 100, 140) 
fig, (ax1) = plt.subplots(1, 1, sharex=True)

ax1.plot(t, filtered3)
ax1.set_title('Data after BandPassFilter')
ax1.axis([0, 1, -2, 2])
ax1.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

super_fft(filtered3) 

#小結


---


1.   頻域訊號能夠有效拆分出時域訊號的成分
2.   濾波處理雖能大程度地修飾訊號，但可能存在一些轉換失真
3.   濾波頻率的選擇會大幅度地影響最後修飾完的訊號樣貌




